# 03 — Comparison on the identical proxy

All arms are scored by the official metric on the same seven degraded matches. Δ is paired by match; intervals are 95% paired bootstrap over matches (1000 draws). With seven matches, intervals are wide: read *decisions*, not third decimals.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp1_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_1')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
import exp1_common as C
import exp1_report as R, exp1_protocol as P
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## Post-hoc diagnostic (added after the frozen run)
The frozen selection chose γ = 0 (constant penalty) in every fold, so `setseq_bd` ≡ `setseq_const`. To measure what the boundary-aware decoder does when it *is* used, γ is forced into {0.5, 1, 2}; everything else stays nested. This arm is a diagnostic, never a candidate.

In [2]:
forced = P.evaluate_variant('setseq_bd_forced', P.ensure_predictions('setseq'))
C.write_json(C.RESULTS / 'variant_setseq_bd_forced.json', forced)
print(pd.DataFrame(forced['rows'])[['match', 'spell_f1', 'n_true', 'n_pred', 'gamma', 'lambda_team', 'lambda_dead']].round(4).to_string(index=False))

 match  spell_f1  n_true  n_pred  gamma  lambda_team  lambda_dead
J03WMX    0.6155     511     326    0.5          2.0         16.0
J03WN1    0.6413     500     347    0.5          2.0          8.0
J03WOH    0.6578     402     301    0.5          2.0          2.0
J03WOY    0.6538     456     365    0.5          2.0          2.0
J03WPY    0.6742     421     323    0.5          2.0          2.0
J03WQQ    0.6810     401     372    0.5          2.0          4.0
J03WR9    0.6168     370     283    0.5          2.0          2.0


In [3]:
parts = [pd.read_csv(C.RESULTS / 'rule_rows.csv'), pd.read_csv(C.RESULTS / 'ryu_rows.csv')]
for variant in ('gbm_const', 'setseq_argmax', 'setseq_const', 'setseq_noaug_bd', 'setseq_bd', 'setseq_bd_forced'):
    parts.append(pd.DataFrame(C.read_json(C.RESULTS / f'variant_{variant}.json')['rows']))
rows = pd.concat(parts, ignore_index=True)
report = R.build_report(rows)
table = pd.DataFrame(report['table'])
print(R.as_markdown(table))

| variant | mean_spell_f1 | mean_macro_f1 | mean_abs_count_error | max_abs_count_error | delta_vs_baseline | ci95_low | ci95_high |
|---|---|---|---|---|---|---|---|
| setseq_const | 0.6556 | 0.9178 | 0.2119 | 0.3425 | +0.4695 | +0.4498 | +0.4871 |
| setseq_bd | 0.6556 | 0.9178 | 0.2119 | 0.3425 | +0.4695 | +0.4498 | +0.4871 |
| setseq_bd_forced | 0.6486 | 0.9177 | 0.2370 | 0.3620 | +0.4625 | +0.4462 | +0.4773 |
| setseq_argmax | 0.6240 | 0.9176 | 0.0889 | 0.1820 | +0.4379 | +0.4105 | +0.4601 |
| setseq_noaug_bd | 0.6005 | 0.8951 | 0.2300 | 0.3523 | +0.4143 | +0.3784 | +0.4441 |
| ryu_final | 0.4874 | 0.8787 | 0.4570 | 0.5284 | +0.3012 | +0.2714 | +0.3269 |
| gbm_const | 0.4577 | 0.8678 | 0.3417 | 0.4325 | +0.2715 | +0.2492 | +0.2912 |
| rasya_c08 | 0.3249 | 0.5915 | 0.4107 | 0.4892 | +0.1388 | +0.1255 | +0.1516 |
| rasya_c06 | 0.3223 | 0.6342 | 0.3571 | 0.4638 | +0.1361 | +0.1281 | +0.1463 |
| baseline | 0.1861 | 0.4579 | 0.1790 | 0.3541 | +0.0000 | +0.0000 | +0.0000 |


## Per-match spellF1 and possession counts

In [4]:
print(R.per_match_matrix(rows).round(4).to_string())
counts = R.per_match_matrix(rows, 'n_pred')
counts.insert(0, 'n_true', R.per_match_matrix(rows, 'n_true')['baseline'])
print(); print(counts.to_string())

variant  baseline  ryu_final  rasya_c06  rasya_c08  gbm_const  setseq_argmax  setseq_const  setseq_noaug_bd  setseq_bd  setseq_bd_forced
match                                                                                                                                   
J03WMX     0.1805     0.4489     0.3103     0.3223     0.4180         0.6006        0.6168           0.5748     0.6168            0.6155
J03WN1     0.1716     0.4043     0.3013     0.3075     0.4020         0.6072        0.6439           0.5914     0.6439            0.6413
J03WOH     0.1898     0.5144     0.3169     0.3179     0.4867         0.6564        0.6750           0.6058     0.6750            0.6578
J03WOY     0.1671     0.4490     0.3163     0.3224     0.4138         0.6383        0.6657           0.6258     0.6657            0.6538
J03WPY     0.1860     0.5088     0.3468     0.3541     0.4826         0.6598        0.6832           0.6514     0.6832            0.6742
J03WQQ     0.2148     0.5422     0.3383  

## Pairwise: main arm vs every other arm, and ablations

In [5]:
print(pd.DataFrame(report['pairwise_main_vs'])[['a', 'b', 'mean_delta', 'ci95', 'wins', 'decision']].to_string(index=False))
print(); print(pd.DataFrame(report['ablations'])[['ablation', 'a', 'b', 'mean_delta', 'ci95', 'wins', 'decision']].to_string(index=False))
print(); print('quality gates (exp_0 protocol):', json.dumps(report['gates'], indent=1))

        a                b  mean_delta                                          ci95  wins     decision
setseq_bd         baseline    0.469454     [0.4497986438677648, 0.48710150197975416]     7    supported
setseq_bd        ryu_final    0.168210     [0.1320836790690651, 0.20188791609007606]     7    supported
setseq_bd        rasya_c06    0.333321       [0.314986301098036, 0.3493622440514427]     7    supported
setseq_bd        rasya_c08    0.330646     [0.30899042672227855, 0.3513562668003685]     7    supported
setseq_bd        gbm_const    0.197925     [0.1690158434335314, 0.22481158194322395]     7    supported
setseq_bd    setseq_argmax    0.031590   [0.022231367049096507, 0.04263753927368148]     7    supported
setseq_bd     setseq_const    0.000000                                    [0.0, 0.0]     0 inconclusive
setseq_bd  setseq_noaug_bd    0.055112    [0.04024843573538184, 0.07344217857791396]     7    supported
setseq_bd setseq_bd_forced    0.006946 [0.0025787511403377946, 0

## Figures (proxy only)

In [6]:
matrix = R.per_match_matrix(rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = list(matrix.mean().sort_values().index)
for name in C.MATCH_IDS:
    axes[0].plot(order, matrix.loc[name, order], marker='o', alpha=0.45, lw=1)
axes[0].plot(order, matrix[order].mean(), color='black', marker='D', lw=2.5, label='mean')
axes[0].set_ylabel('spellF1 (held-out match)'); axes[0].set_title('Per-match spellF1 by arm (lines = matches)')
axes[0].tick_params(axis='x', rotation=35); axes[0].legend()
ratio = R.per_match_matrix(rows, 'count_error_ratio')[order]
axes[1].axhspan(-0.3, 0.3, color='0.9', label='+-30% gate')
for name in C.MATCH_IDS:
    axes[1].plot(order, ratio.loc[name], marker='o', alpha=0.45, lw=1)
axes[1].axhline(0, color='black', lw=1); axes[1].set_ylabel('predicted / true possessions - 1')
axes[1].set_title('Over/under-segmentation'); axes[1].tick_params(axis='x', rotation=35); axes[1].legend()
fig.tight_layout(); fig.savefig(C.RESULTS / 'fig_proxy_comparison.png', dpi=130)
print('saved', C.RESULTS / 'fig_proxy_comparison.png')

saved /home/jeremy-mboe/Documents/Kuliah/Sem5/barca/exp_jeremy/exp_1/results/fig_proxy_comparison.png
